[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/ThanhDatVN/AI_journey/blob/main/notebooks/22_anomaly_detection/90_exercises.ipynb)

# Bài tập — Chương 22: Anomaly Detection (L0 → L7)

In [ ]:
# ⚙️ Setup — chạy được cả local lẫn Google Colab (local: đã `pip install -e .` thì bỏ qua)
import sys
if "google.colab" in sys.modules:
    !test -d /content/ml-course || git clone -q --depth 1 https://github.com/ThanhDatVN/AI_journey /content/ml-course
    !pip install -q -e /content/ml-course
    sys.path.insert(0, "/content/ml-course/src")
    %cd /content/ml-course/notebooks/22_anomaly_detection

In [ ]:
import matplotlib.pyplot as plt
import numpy as np
from scipy.stats import chi2, multivariate_normal, norm, spearmanr
from sklearn.covariance import MinCovDet
from sklearn.ensemble import IsolationForest
from sklearn.metrics import average_precision_score, f1_score, roc_auc_score

from ml_course.checks import check_close, check_equal, check_greater, check_less, check_true

rng = np.random.default_rng(0)


def make_servers(n_normal, n_anom, seed):
    """Máy chủ: (thông lượng MB/s, độ trễ ms). Nhãn 1 = sự cố."""
    r = np.random.default_rng(seed)
    normal = r.multivariate_normal([14, 15], [[1.6, 0.4], [0.4, 1.4]], n_normal)
    anomalies = np.c_[r.uniform(5, 25, 3 * n_anom), r.uniform(5, 25, 3 * n_anom)]
    anomalies = anomalies[np.linalg.norm(anomalies - [14, 15], axis=1) > 2.5][:n_anom]
    X = np.vstack([normal, anomalies])
    return X, np.r_[np.zeros(len(normal), int), np.ones(len(anomalies), int)]

## L0 — Nhớ khái niệm

**Câu 1.** (a) Ba loại bất thường, mỗi loại một ví dụ. (b) Công thức $p(\mathbf x)$ của mô hình Gaussian (feature độc lập) và quy tắc cảnh báo. (c) Vì sao chọn ε bằng F1 thay vì accuracy? (d) Công thức anomaly score của Isolation Forest; giá trị gần 1 và gần 0.5 nghĩa là gì?

✍️ Trả lời:

## L1 — Giải thích bằng lời

**Câu 2.** Đội chống gian lận có model HistGradientBoosting có giám sát đạt PR-AUC 0.9 trên tập test (gồm các vụ gian lận năm qua), trong khi Isolation Forest chạy song song chỉ đạt 0.6. Trưởng nhóm đề xuất tắt Isolation Forest "vì model có giám sát tốt hơn hẳn". Bạn đồng ý không? Phản biện và đề xuất cách đánh giá công bằng hơn.

✍️ Trả lời:

## L2 — Tính tay

**Câu 3.** Độ trễ (ms) của 9 request: `[100, 102, 98, 101, 99, 103, 97, 100, 250]`.
(a) Trung bình, độ lệch chuẩn (chia $n$), z-score của 250. Quy tắc $|z| > 3$ có bắt được không?
(b) Median, MAD, robust z-score $0.6745\,(x - \text{median})/\text{MAD}$ của 250.
(c) Kiểm tra bằng số: với $n$ điểm, $|z|$ (độ lệch chuẩn chia $n$) không bao giờ vượt $\sqrt{n - 1}$ — vì sao điều này làm z-score vô dụng với mẫu nhỏ?

In [ ]:
x3 = np.array([100, 102, 98, 101, 99, 103, 97, 100, 250.0])
### START CODE HERE ### (≈ 5 dòng)
z_250 = None
median3 = None
mad3 = None
robust_z_250 = None
max_possible_z = None
### END CODE HERE ###
print(f"z = {z_250:.3f} (giới hạn {max_possible_z:.3f}) | median = {median3}, MAD = {mad3}, robust z = {robust_z_250:.2f}")

In [ ]:
check_close(z_250, 2.8264, tol=1e-3, message="z-score của 250")
check_less(z_250, 3, "|z| > 3 không bắt được — outlier thổi phồng std")
check_close(median3, 100, message="median")
check_close(mad3, 2, message="MAD")
check_close(robust_z_250, 50.5875, tol=1e-3, message="robust z-score")
check_true(max_possible_z - 0.01 < z_250 < max_possible_z, "một outlier lớn gần chạm (nhưng không vượt) giới hạn √(n−1)")

**Câu 4.** Mô hình Gaussian độc lập: $\boldsymbol\mu = [10, 50]$, $\boldsymbol\sigma^2 = [4, 25]$. Điểm $\mathbf x = [13, 40]$.
(a) Tính $p(\mathbf x)$. Với $\varepsilon = 10^{-3}$, có cảnh báo không?
(b) Bình phương khoảng cách Mahalanobis $d^2 = \sum_j (x_j - \mu_j)^2/\sigma_j^2$.
(c) Dưới giả định Gaussian, $d^2 \sim \chi^2_2$; tính $P(\chi^2_2 > d^2) = e^{-d^2/2}$ — xác suất một điểm bình thường lệch xa ít nhất như vậy.

In [ ]:
### START CODE HERE ### (≈ 4 dòng)
p_x4 = None
is_anomaly4 = None
d2_4 = None
tail4 = None
### END CODE HERE ###
print(f"p(x) = {p_x4:.3e} | cảnh báo: {is_anomaly4} | d² = {d2_4} | P(χ²₂ > d²) = {tail4:.4f}")

In [ ]:
check_close(p_x4, norm.pdf(13, 10, 2) * norm.pdf(40, 50, 5), tol=1e-9, message="p(x)")
check_true(is_anomaly4, "p(x) ≈ 7e-4 < 1e-3 → cảnh báo")
check_close(d2_4, 6.25, message="Mahalanobis²")
check_close(tail4, chi2.sf(6.25, 2), tol=1e-6, message="xác suất đuôi χ²₂")

## L3 — Code

**Bài 5.** (Theo khoá Machine Learning của Andrew Ng.) Cài đặt:
- `estimate_gaussian(X)` → `(mu, var)` (phương sai chia $n$);
- `gaussian_pdf(X, mu, Sigma)`: nếu `Sigma` là vector → coi như ma trận chéo; trả về mật độ Gaussian nhiều chiều của từng dòng;
- `select_threshold(y_val, p_val)` → `(best_eps, best_f1)`: thử 1000 giá trị ε đều nhau giữa min và max của `p_val`, cảnh báo khi $p < \varepsilon$.

Áp dụng: ước lượng trên `X5_train` (chỉ bình thường), chọn ε trên validation, đo F1 trên test (`f1_5`).

In [ ]:
def estimate_gaussian(X):
    ### START CODE HERE ### (≈ 1 dòng)
    return None, None   # trung bình và phương sai (chia n) theo cột
    ### END CODE HERE ###


def gaussian_pdf(X, mu, Sigma):
    ### START CODE HERE ### (≈ 6 dòng)
    # Sigma 1-D → np.diag(Sigma); Xc = X − mu
    # p = (2π)^(−d/2) |Σ|^(−1/2) exp(−½ · Xc Σ⁻¹ Xcᵀ) — chỉ lấy đường chéo (dùng einsum hoặc sum)
    pass
    ### END CODE HERE ###


def select_threshold(y_val, p_val):
    best_eps, best_f1 = 0.0, 0.0
    ### START CODE HERE ### (≈ 5 dòng)
    # duyệt eps trong np.linspace(min, max, 1000); pred = p_val < eps; F1 (zero_division=0); giữ eps có F1 lớn nhất
    pass
    ### END CODE HERE ###
    return best_eps, best_f1

In [ ]:
X5_train, _ = make_servers(1000, 0, seed=11)
X5_val, y5_val = make_servers(300, 30, seed=12)
X5_test, y5_test = make_servers(300, 30, seed=13)
### START CODE HERE ### (≈ 4 dòng)
mu5, var5 = None, None       # ước lượng trên X5_train
eps5, f1_val5 = None, None   # select_threshold trên validation
f1_5 = None                  # F1 trên test với eps5
### END CODE HERE ###
print(f"ε = {eps5:.2e} | F1 validation = {f1_val5:.3f} | F1 test = {f1_5:.3f}")

In [ ]:
check_close(mu5, X5_train.mean(0), message="mu")
check_close(var5, X5_train.var(0), message="var chia n")
S = np.array([[2.0, 0.6], [0.6, 1.0]])
check_close(gaussian_pdf(X5_val[:20], mu5, S), multivariate_normal(mu5, S).pdf(X5_val[:20]), tol=1e-12, message="pdf với Σ đầy đủ")
check_close(gaussian_pdf(X5_val[:20], mu5, var5), multivariate_normal(mu5, np.diag(var5)).pdf(X5_val[:20]), tol=1e-12, message="pdf với Σ chéo")
e, f = select_threshold(np.array([0, 0, 0, 1, 1]), np.array([0.9, 0.8, 0.7, 0.1, 0.2]))
check_close(f, 1.0, message="ngưỡng tách hoàn hảo cho F1 = 1")
check_true(0.2 < e <= 0.7, "ε nằm giữa điểm bất thường và bình thường")
check_greater(f1_5, 0.8, "F1 test > 0.8")

**Bài 6.** Viết `mahalanobis_sq(X, mu, cov)` → $d^2$ của từng dòng. Dưới giả định Gaussian $d$ chiều, $d^2 \sim \chi^2_d$ → ngưỡng "1 phần nghìn" là `chi2.ppf(0.999, d)`. Kiểm tra trên 20 000 điểm Gaussian 4 chiều tương quan (tỉ lệ bị cảnh báo `rate6` phải ≈ 0.001) và 3 điểm "kết hợp lạ" (`flags_odd6`).

In [ ]:
def mahalanobis_sq(X, mu, cov):
    ### START CODE HERE ### (≈ 2 dòng)
    return None   # Xc = X − mu; tổng theo hàng của (Xc @ cov⁻¹) * Xc
    ### END CODE HERE ###

In [ ]:
cov6 = np.array([[1, 0.8, 0.5, 0], [0.8, 1, 0.4, 0], [0.5, 0.4, 1, 0], [0, 0, 0, 1.0]])
X6 = rng.multivariate_normal(np.zeros(4), cov6, 20000)
odd6 = np.array([[2, -2, 0, 0], [-1.8, 1.5, 0, 0], [1.5, -1.5, 0, 0]])       # từng giá trị ≤ 2σ, kết hợp trái tương quan 0.8
### START CODE HERE ### (≈ 4 dòng)
mu6, c6 = None, None    # trung bình và hiệp phương sai của X6
thr6 = None             # chi2.ppf(0.999, số chiều)
rate6 = None            # tỉ lệ điểm X6 có d² > thr6
flags_odd6 = None       # mảng bool cho 3 điểm odd6
### END CODE HERE ###
print(f"ngưỡng χ²₄(0.999) = {thr6:.2f} | tỉ lệ cảnh báo trên dữ liệu bình thường = {rate6:.4f} | 3 điểm lạ: {flags_odd6}")

In [ ]:
d2_ref = MinCovDet(random_state=0).fit(X6[:2000])
check_close(mahalanobis_sq(X6[:50], d2_ref.location_, d2_ref.covariance_), d2_ref.mahalanobis(X6[:50]), tol=1e-8, message="khớp sklearn")
check_true(0.0003 < rate6 < 0.0025, "tỉ lệ cảnh báo ≈ 0.1% như χ² dự đoán")
check_true(flags_odd6.all(), "cả 3 điểm trái tương quan đều bị cảnh báo")
check_true((np.abs(odd6) <= 2).all(), "dù từng giá trị không vượt 2σ")

## L4 — Debug

**Bài 7.** Đoạn code sau xây detector Gaussian nhưng có 3 lỗi: (1) ước lượng $\mu, \sigma^2$ trên **validation** (có lẫn bất thường) thay vì dữ liệu train bình thường; (2) chọn ε theo **accuracy**; (3) báo cáo F1 trên chính validation dùng để chọn ε (lạc quan). Sửa và lưu `eps7`, `f1_7` (F1 trên **test**).

In [ ]:
X7_train, _ = make_servers(1000, 0, seed=21)
X7_val, y7_val = make_servers(300, 30, seed=22)
X7_test, y7_test = make_servers(300, 30, seed=23)
### START CODE HERE ### (≈ 4 dòng)
mu7, var7 = X7_val.mean(0), X7_val.var(0)                                            # 🐛 (1)
p7 = gaussian_pdf(X7_val, mu7, var7)
eps7 = max(np.linspace(p7.min(), p7.max(), 1000), key=lambda e: ((p7 < e) == y7_val).mean())  # 🐛 (2)
f1_7 = f1_score(y7_val, p7 < eps7)                                                     # 🐛 (3)
### END CODE HERE ###
print(f"ε = {eps7:.2e} | F1 test = {f1_7:.3f}")

In [ ]:
check_greater(f1_7, 0.8, "F1 test > 0.8")
check_close(eps7, select_threshold(y7_val, gaussian_pdf(X7_val, *estimate_gaussian(X7_train)))[0], message="ε chọn bằng F1 trên validation, μ/σ² từ train")

## L5 — Phân tích

**Bài 8.** **Dữ liệu train bị nhiễm bẩn.** Dữ liệu bình thường 2D tương quan 0.8; bất thường là một cụm quanh $(2.5, -2.5)$ (cùng một kiểu lỗi lặp lại). Train trên dữ liệu có lẫn tỉ lệ $c \in \{0, 0.02, 0.05, 0.1, 0.2\}$ bất thường (không nhãn); đánh giá PR-AUC trên test (2% bất thường) cho:
- `ap_classic[c]`: Mahalanobis với trung bình/hiệp phương sai thường;
- `ap_robust[c]`: Mahalanobis với `MinCovDet(random_state=0)`.

Nhận xét: vì sao detector cổ điển suy giảm khi $c$ tăng? Khi nào ngay cả MCD cũng thất bại?

In [ ]:
cov8 = np.array([[1, 0.8], [0.8, 1.0]])


def sample8(n, frac, seed):
    r = np.random.default_rng(seed)
    n_bad = int(round(frac * n))
    X = np.vstack([r.multivariate_normal([0, 0], cov8, n - n_bad), r.normal([2.5, -2.5], 0.4, (n_bad, 2))])
    return X, np.r_[np.zeros(n - n_bad, int), np.ones(n_bad, int)]


X8_test, y8_test = sample8(5000, 0.02, seed=99)
ap_classic, ap_robust = {}, {}
### START CODE HERE ### (≈ 6 dòng)
# với mỗi c: X_tr, _ = sample8(2000, c, seed=1)
# classic: mahalanobis_sq(X8_test, mean, np.cov); robust: MinCovDet(random_state=0).fit(X_tr).mahalanobis(X8_test)
pass
### END CODE HERE ###
for c in ap_classic:
    print(f"c = {c:<5}: PR-AUC classic = {ap_classic[c]:.3f} | robust (MCD) = {ap_robust[c]:.3f}")

In [ ]:
check_greater(ap_classic[0], 0.9, "không nhiễm bẩn: classic tốt")
check_less(ap_classic[0.2], ap_classic[0] - 0.3, "classic suy giảm mạnh khi 20% nhiễm bẩn")
check_greater(ap_robust[0.2], 0.9, "MCD vẫn tốt khi 20% nhiễm bẩn")

## L6 — Thực tế

**Bài 9.** **Giám sát máy chủ với 6 chỉ số** (CPU %, RAM %, mạng vào, mạng ra, độ trễ, tỉ lệ lỗi) — các chỉ số tương quan qua tải hệ thống. Ba kiểu sự cố: **vòng lặp vô hạn** (CPU cao bất thường *so với* lưu lượng mạng — từng chỉ số vẫn trong khoảng thường gặp), **rò rỉ bộ nhớ** (RAM rất cao), **tấn công DDoS** (mạng vào và độ trễ rất cao).

(a) Gaussian nhiều chiều, ε chọn bằng `select_threshold` trên **p** như Bài 5 → F1 test `f1_multi_linear`.
(b) Trong 6 chiều, $p(\mathbf x)$ trải dài nhiều bậc độ lớn → lưới 1000 giá trị ε đều nhau giữa min và max của $p$ quá thô ở vùng $p$ nhỏ (nơi ngưỡng tốt nằm). Chọn ngưỡng trên **log p** (`select_threshold(y, np.log(p))` — log đơn điệu nên quy tắc "p < ε" giữ nguyên) → `f1_multi`.
(c) Mô hình Gaussian độc lập, cũng chọn ngưỡng trên log p → `f1_indep`.
(d) Recall riêng kiểu "vòng lặp" của hai mô hình (b), (c): `recall_loop_multi`, `recall_loop_indep`.

In [ ]:
def make_server_metrics(n_normal, n_per_type, seed):
    r = np.random.default_rng(seed)
    n = n_normal + 3 * n_per_type
    load = r.normal(0, 1, n)
    X = np.c_[50 + 15 * load + r.normal(0, 3, n),                 # CPU %
              60 + r.normal(0, 5, n),                              # RAM %
              100 + 30 * load + r.normal(0, 5, n),                 # mạng vào
              80 + 24 * load + r.normal(0, 5, n),                  # mạng ra
              80 + 10 * load + r.normal(0, 4, n),                  # độ trễ ms
              1 + 0.2 * load + r.normal(0, 0.15, n)]               # tỉ lệ lỗi %
    kind = np.r_[np.zeros(n_normal, int), np.repeat([1, 2, 3], n_per_type)]
    loop, leak, ddos = kind == 1, kind == 2, kind == 3
    X[loop, 0] = 50 + 15 * (load[loop] + 1.6) + r.normal(0, 3, loop.sum())    # CPU như tải +1.6σ, mạng vẫn theo tải thật
    X[leak, 1] += r.uniform(25, 35, leak.sum())
    X[ddos, 2] += r.uniform(120, 180, ddos.sum())
    X[ddos, 4] += r.uniform(40, 60, ddos.sum())
    return X, (kind > 0).astype(int), kind


X9_train, _, _ = make_server_metrics(5000, 0, seed=31)
X9_val, y9_val, k9_val = make_server_metrics(1000, 10, seed=32)
X9_test, y9_test, k9_test = make_server_metrics(1000, 10, seed=33)
### START CODE HERE ### (≈ 12 dòng)
# mu9, var9 từ train; cov9 = np.cov(X9_train.T)
# (a) ngưỡng trên p; (b), (c) ngưỡng trên np.log(p); F1 và recall kiểu 1 (k9_test == 1) trên test
f1_multi_linear = None
f1_multi, f1_indep = None, None
recall_loop_multi, recall_loop_indep = None, None
### END CODE HERE ###
print(f"Gaussian nhiều chiều — F1 test: lưới ε trên p = {f1_multi_linear:.3f} | lưới trên log p = {f1_multi:.3f}")
print(f"Gaussian độc lập (log p) — F1 test = {f1_indep:.3f}")
print(f"recall kiểu vòng lặp: độc lập = {recall_loop_indep:.2f} | nhiều chiều = {recall_loop_multi:.2f}")

In [ ]:
check_greater(f1_multi, f1_multi_linear + 0.1, "chọn ngưỡng trên log p tốt hơn rõ lưới trên p")
check_greater(f1_multi, 0.9, "Gaussian nhiều chiều (log p) F1 > 0.9")
check_greater(f1_multi, f1_indep + 0.2, "nhiều chiều vượt rõ độc lập")
check_greater(recall_loop_multi, 0.8, "nhiều chiều bắt được vòng lặp vô hạn")
check_less(recall_loop_indep, 0.5, "độc lập bỏ sót phần lớn vòng lặp vô hạn")

## L7 — Nâng cao

**Bài 10.** Cài đặt `IsolationForestScratch(n_trees, sample_size, seed)`:
- `fit(X)`: với mỗi cây, lấy mẫu con `sample_size` điểm (không lặp), xây cây cô lập đệ quy với giới hạn chiều cao $\lceil\log_2 \psi\rceil$: chọn feature ngẫu nhiên, ngưỡng đều trong $[\min, \max]$; dừng khi nút còn ≤ 1 điểm, hết chiều cao, hoặc mọi giá trị bằng nhau.
- `path_length(x, node, depth)`: đi xuống tới lá; tại lá cộng thêm $c(\text{size})$ (ước lượng phần cây chưa xây).
- `score_samples(X)` → $2^{-\bar h(x)/c(\psi)}$ (lớn = bất thường).

So sánh với sklearn: ROC-AUC và tương quan hạng (Spearman).

In [ ]:
def c_factor(n):
    if n <= 1:
        return 0.0
    return 2 * (np.log(n - 1) + 0.5772156649) - 2 * (n - 1) / n


class IsolationForestScratch:
    def __init__(self, n_trees=100, sample_size=256, seed=0):
        self.n_trees, self.sample_size, self.seed = n_trees, sample_size, seed

    def _build(self, X, depth, r):
        ### START CODE HERE ### (≈ 9 dòng)
        # lá: {"size": len(X)} khi len(X) <= 1 hoặc depth >= self.height_limit_ hoặc không còn feature nào có max > min
        # nút trong: j ngẫu nhiên trong các feature có max > min, t ~ U(min, max); trả về dict {"j", "t", "left", "right"}
        pass
        ### END CODE HERE ###

    def fit(self, X):
        r = np.random.default_rng(self.seed)
        X = np.asarray(X, float)
        self.psi_ = min(self.sample_size, len(X))
        self.height_limit_ = int(np.ceil(np.log2(self.psi_)))
        self.trees_ = [self._build(X[r.choice(len(X), self.psi_, replace=False)], 0, r) for _ in range(self.n_trees)]
        return self

    def path_length(self, x, node, depth=0):
        ### START CODE HERE ### (≈ 4 dòng)
        # lá ("size" in node): depth + c_factor(node["size"]); ngược lại đi trái nếu x[j] < t, phải nếu không
        pass
        ### END CODE HERE ###

    def score_samples(self, X):
        ### START CODE HERE ### (≈ 2 dòng)
        # h̄ = trung bình path_length qua các cây; return 2 ** (−h̄ / c_factor(psi_))
        pass
        ### END CODE HERE ###

In [ ]:
X10 = np.vstack([rng.normal(0, 1, (500, 3)), rng.uniform(-6, 6, (20, 3))])
y10 = np.r_[np.zeros(500), np.ones(20)]
mine10 = IsolationForestScratch(n_trees=100, sample_size=256, seed=0).fit(X10).score_samples(X10)
sk10 = -IsolationForest(n_estimators=100, max_samples=256, random_state=0).fit(X10).score_samples(X10)
auc_mine, auc_sk = roc_auc_score(y10, mine10), roc_auc_score(y10, sk10)
rho10 = spearmanr(mine10, sk10).statistic
print(f"ROC-AUC: scratch = {auc_mine:.3f} | sklearn = {auc_sk:.3f} | Spearman(scratch, sklearn) = {rho10:.3f}")
print(f"điểm TB: bình thường = {mine10[y10 == 0].mean():.3f} | bất thường = {mine10[y10 == 1].mean():.3f}")

In [ ]:
check_true(((mine10 > 0) & (mine10 <= 1)).all(), "điểm ∈ (0, 1]")
check_greater(auc_mine, 0.9, "ROC-AUC scratch > 0.9")
check_greater(rho10, 0.9, "xếp hạng gần như trùng sklearn")
check_greater(mine10[y10 == 1].mean(), mine10[y10 == 0].mean() + 0.15, "bất thường có điểm cao hơn rõ")